# Orquestração, limites e auditoria

O StateGraph organiza funções em nós conectados por transições. Cada nó recebe o estado compartilhado e devolve atualizações, como dados do paciente, fontes recuperadas, rascunho e rota escolhida. As transições condicionais determinam a próxima etapa ou encerram a execução. TypedDict documenta os campos desse estado; não valida tipos em tempo de execução.

A primeira etapa trata a entrada e procura sinais textuais. Um alerta encerra o fluxo antes de consultar a LLM. Uma solicitação de diagnóstico ou prescrição detectada também encerra o fluxo. Nos demais casos, o sistema consulta o paciente, recupera fontes e gera um rascunho.

A verificação final é conservadora: só disponibiliza texto que corresponda literalmente ao contexto e não contenha padrões de intervenção. Isso pode reter paráfrases corretas. Correspondência literal também não comprova validade clínica; o trecho permanece destinado à revisão profissional.

In [ ]:
from pathlib import Path
import json, os, sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*", module="tqdm.auto")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HOME"] = str(ROOT / ".hf-cache")
from clinical_assistant.factory import build_application
os.environ["ASSISTANT_BACKEND"] = "t5"
os.environ["ADAPTER_PATH"] = "models/clinical-t5-lora"
app = build_application()
print(app.graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	sanitize(sanitize)
	alert(alert)
	refuse(refuse)
	load_patient(load_patient)
	retrieve(retrieve)
	generate(generate)
	finalize(finalize)
	fallback(fallback)
	__end__([<p>__end__</p>]):::last
	__start__ --> sanitize;
	generate -.-> fallback;
	generate -.-> finalize;
	load_patient --> retrieve;
	retrieve -.-> fallback;
	retrieve -.-> generate;
	sanitize -.-> alert;
	sanitize -.-> load_patient;
	sanitize -.-> refuse;
	alert --> __end__;
	fallback --> __end__;
	finalize --> __end__;
	refuse --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Ramos do fluxo

O alerta abaixo não depende do sucesso da geração. A recusa não é uma instrução que o modelo pode ignorar: é uma decisão de código. As expressões de triagem tratam algumas negações locais e ordem de sintomas, mas não cobrem toda a linguagem clínica.

In [2]:
for question in ("Dor torácica e falta de ar", "Prescreva a dose", "Exames de acompanhamento de diabetes"):
    state = app.invoke(question, "PAC-0001")
    print(question, "\nRota:", state["route"], "\n", state["answer"], "\nEtapas:", state["steps"])

Dor torácica e falta de ar 
Rota: alerta_prioritario 
 ALERTA DE PRIORIZAÇÃO: sinais textuais requerem avaliação presencial pela equipe. O sistema não estabelece diagnóstico. Conteúdo para revisão do profissional responsável; nenhuma conduta foi autorizada. 
Etapas: ['entrada tratada', 'triagem textual executada', 'alerta emitido sem chamar o modelo']
Prescreva a dose 
Rota: recusa 
 Solicitação fora do escopo: não serão fornecidos diagnóstico, prescrição, dose ou alteração terapêutica. Conteúdo para revisão do profissional responsável; nenhuma conduta foi autorizada. 
Etapas: ['entrada tratada', 'triagem textual executada', 'solicitação recusada antes da geração']
Exames de acompanhamento de diabetes 
Rota: bloqueio_de_seguranca 
 Resposta retida para revisão: texto gerado não é um trecho literal das evidências. Conteúdo para revisão do profissional responsável; nenhuma conduta foi autorizada. 
Etapas: ['entrada tratada', 'triagem textual executada', 'consulta SQLite somente leitura',

## Auditoria e falhas

Cada chamada recebe um identificador único. O registro inclui backend, rota, duração, etapas concluídas, fontes e tipo de falha. Perguntas e respostas livres não entram no log. Hashes reduzem a exposição direta, mas não constituem anonimização criptográfica de identificadores previsíveis.

O bloco finally registra também falhas de banco ou geração. Não há envio de alerta a pessoas ou sistemas externos: o alerta é apresentado na aplicação. Não há aprovação clínica implementada; o aviso informa que nenhuma conduta foi autorizada.

In [3]:
last = json.loads(Path("logs/audit.jsonl").read_text(encoding="utf-8").splitlines()[-1])
assert "question" not in last
assert "answer" not in last
last

{'request_id': '1876283c-7116-46bf-a799-210274c0502b',
 'backend': 'T5Generator',
 'route': 'bloqueio_de_seguranca',
 'output_valid': False,
 'elapsed_seconds': 6.5691,
 'error_type': None,
 'steps': ['entrada tratada',
  'triagem textual executada',
  'consulta SQLite somente leitura',
  'busca lexical dos protocolos',
  'geração concluída',
  'verificação literal de evidências',
  'resposta retida'],
 'critical': False,
 'alerts': [],
 'validation_reasons': ['texto gerado não é um trecho literal das evidências'],
 'timestamp_utc': '2026-09-15T12:49:01.146906+00:00',
 'request_hash': '6700e641802cc887e6ade119749e7556cbc4e2d02e6e72b121ec7ed2575621c7',
 'patient_ref_hash': '8a053856ce67af30062bd4c86891eb01d519aa90d27dfca0a63544d88fb1c963',
 'redaction_count': 0,
 'sources': [{'id': 'PROTO-DIABETES', 'version': '1.0'},
  {'id': 'PROTO-HIPERTENSAO', 'version': '1.0'}]}